# Project — TGV punctuality by route — *Régularité mensuelle TGV par liaisons*
**MSA-DATI07-03 · Advanced EDA & Data Collection** · difficulty ⭐ easy

This is your **project notebook**. It already follows the six-section structure from Session 6 and contains working collection code for this API. Your job: fill every `# TODO` and every *TODO* markdown cell, on **real data you collected yourselves**.

| # | Section | Answers the question… |
|---|---|---|
| 1 | Context & provenance | Where does this data come from, and what question is it for? |
| 2 | Setup & parameters | Can someone else run this and get the same thing? |
| 3 | Collection, ingestion & schema | Did we collect it properly, and is it what we think it is? |
| 4 | Data quality | What is missing, duplicated or weird — and what did we decide? |
| 5 | Exploration | What does the data say? Every figure captioned. |
| 6 | Findings & next steps | What can a modeller trust, and what should they test? |

**Deliverables:** this notebook (Restart & Run All passes, same fingerprint twice) · `requirements.txt` · the 2-page data-quality report (`DATA_QUALITY_REPORT_TEMPLATE.md`) · regular Git commits.

**Possible questions** (pick one or write your own — one per group):
- Which routes are the least punctual, and is it getting better or worse over time?
- What share of delays comes from external causes vs infrastructure vs rolling stock?
- Do long routes have more delays, or just longer ones?


---
## 1 · Context & provenance

*TODO — complete this card before writing any analysis.*

| Field | Your answer |
|---|---|
| **Business question** | *TODO* |
| **Source** | SNCF open-data portal (ressources.data.sncf.com), dataset `regularite-mensuelle-tgv-aqst` |
| **Dataset page / API docs** | https://ressources.data.sncf.com/explore/dataset/regularite-mensuelle-tgv-aqst/ |
| **Collected by / when** | *TODO: group name, date of your collection run* |
| **Collection method** | REST API, paginated, cached with `requests-cache` (see Section 3) |
| **Filters applied** | *TODO: copy from the parameters cell* |
| **Licence / constraints** | ODbL (TO CONFIRM on the dataset page) — check on the dataset page |
| **Unit of observation** | one row = one route (departure → arrival station) for one month (TO CONFIRM) |
| **Known limitations** | *TODO: what the API does not give you, caps, snapshots, filters…* |

---
## 2 · Setup & parameters
On Colab, run the install cell once per session.

In [1]:
# Colab / fresh environment only
import importlib.util
missing = [p for p in ("pandera", "requests_cache") if importlib.util.find_spec(p) is None]
if missing:
    !pip install -q "pandera>=0.24" requests-cache

In [2]:
# --- Parameters: the ONLY cell you should need to edit to rerun on new data ---
SEED = 42
DATA_DIR = "data"
REPORT_DIR = "reports"
CACHE_HOURS = 24 * 7                              # re-use API responses for a week
IQR_K = 1.5
MAX_MISSING_SHARE = 0.30

BASE_URL = "https://ressources.data.sncf.com"
DATASET_ID = "regularite-mensuelle-tgv-aqst"
WHERE = None
ORDER_BY = "date"
MAX_ROWS = 10_000
PAUSE_S = 0.5

In [3]:
import hashlib, json, random, time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pandera
import pandera.pandas as pa
import requests
import requests_cache
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

random.seed(SEED); np.random.seed(SEED)
DATA, REPORTS = Path(DATA_DIR), Path(REPORT_DIR)
for p in (DATA / "raw", DATA / "processed", REPORTS):
    p.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

for lib in (pd, np, sns, pandera, requests):
    print(f"{lib.__name__:<10}{lib.__version__}")

pandas    2.1.4
numpy     1.26.4
seaborn   0.12.2
pandera   0.35.0
requests  2.31.0


In [4]:
# Write requirements.txt from the packages this notebook imports
from importlib.metadata import version
pkgs = ["pandas", "numpy", "matplotlib", "seaborn", "pandera", "pyarrow", "requests", "requests-cache"]
Path("requirements.txt").write_text("".join(f"{p}=={version(p)}\n" for p in pkgs))
print(Path("requirements.txt").read_text())

pandas==2.1.4
numpy==1.26.4
matplotlib==3.8.0
seaborn==0.12.2
pandera==0.35.0
pyarrow==14.0.2
requests==2.31.0
requests-cache==1.3.3



---
## 3 · Collection, ingestion & schema

### 3.1 A polite, cached HTTP session
- **Cache**: the same request is never sent twice (fast reruns, kind to the API).
- **Retry with backoff** on 429 / 5xx, respecting `Retry-After`.
- **Throttle**: pause between requests that actually hit the network.

In [5]:
session = requests_cache.CachedSession("cache/api", expire_after=CACHE_HOURS * 3600)
session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504],
    respect_retry_after_header=True)))
session.headers["User-Agent"] = "MSA-DATI07-student-project/1.0"

def get(url, params=None):
    r = session.get(url, params=params, timeout=30)
    r.raise_for_status()
    if not getattr(r, "from_cache", False):
        time.sleep(PAUSE_S)
    return r

### 3.2 Collect

In [6]:
def collect_ods(base_url, dataset_id, where=None, order_by=None, max_rows=MAX_ROWS, use_export=False):
    """Opendatasoft Explore API v2.1. Offset pagination, 100 rows per page.
    The API refuses offset + limit > 10,000: above that, filter with `where` or use the export endpoint."""
    url = f"{base_url}/api/explore/v2.1/catalog/datasets/{dataset_id}"
    params = {k: v for k, v in {"where": where, "order_by": order_by}.items() if v}
    if use_export:                                   # one call, no row cap, returns a JSON list
        r = get(f"{url}/exports/json", params=params)
        return r.json()[:max_rows]
    rows, offset, limit = [], 0, 100
    while offset < min(max_rows, 10_000):
        r = get(f"{url}/records", params={**params, "limit": limit, "offset": offset})
        page = r.json()
        if offset == 0:
            print(f"API reports {page['total_count']:,} matching rows")
        rows += page["results"]
        if len(page["results"]) < limit:             # last page
            break
        offset += limit
    return rows[:max_rows]

In [7]:
collected_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
raw_rows = collect_ods(BASE_URL, DATASET_ID, where=WHERE, order_by=ORDER_BY)
print(f"collected {len(raw_rows):,} rows at {collected_at}")

API reports 12,907 matching rows
collected 10,000 rows at 2026-11-04T10:36:36+00:00


### 3.3 Flatten, save raw Parquet + README

In [8]:
df_raw = pd.json_normalize(raw_rows, sep='_')

In [9]:
raw_path = DATA / "raw" / "raw.parquet"
df_raw.to_parquet(raw_path, index=False)
(DATA / "raw" / "README.md").write_text(f"""# Raw data
- Source: see Section 1 of the notebook
- Collected at (UTC): {collected_at}
- Rows x columns: {df_raw.shape[0]} x {df_raw.shape[1]}
- Columns: {", ".join(df_raw.columns)}
""")
print(df_raw.shape)
df_raw.head()

(10000, 26)


,date,service,gare_depart,gare_arrivee,duree_moyenne,nb_train_prevu,nb_annulation,commentaire_annulation,nb_train_depart_retard,retard_moyen_depart,...,nb_train_retard_sup_15,retard_moyen_trains_retard_sup15,nb_train_retard_sup_30,nb_train_retard_sup_60,prct_cause_externe,prct_cause_infra,prct_cause_gestion_trafic,prct_cause_materiel_roulant,prct_cause_gestion_gare,prct_cause_prise_en_charge_voyageurs
0,2018-01,National,BORDEAUX ST JEAN,PARIS MONTPARNASSE,141,870,5,None,289,11.247809,...,110,6.511118,44,8,36.134454,31.092437,10.924370,15.966387,5.042017,0.840336
1,2018-01,National,LE MANS,PARIS MONTPARNASSE,56,406,1,None,213,8.479969,...,32,5.363539,9,4,20.000000,35.000000,16.666667,16.666667,8.333333,3.333333
2,2018-01,National,PARIS MONTPARNASSE,LA ROCHELLE VILLE,166,226,0,None,21,6.239683,...,11,2.938053,6,1,22.222222,27.777778,16.666667,16.666667,5.555556,11.111111
3,2018-01,National,PARIS MONTPARNASSE,NANTES,124,508,3,None,71,7.235211,...,39,5.292211,18,8,33.333333,22.222222,16.666667,20.370370,5.555556,1.851852
4,2018-01,National,POITIERS,PARIS MONTPARNASSE,94,472,4,None,224,6.784673,...,42,4.882372,10,0,15.789474,45.614035,19.298246,15.789474,1.754386,1.754386


In [10]:
# First look: types, missing share, number of distinct values per column
overview = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "share_missing": df_raw.isna().mean().round(3),
    "n_unique": df_raw.nunique(dropna=True),
}).sort_values("share_missing", ascending=False)
overview

,dtype,share_missing,n_unique
commentaire_annulation,object,1.00,0
commentaire_retards_depart,object,1.00,0
commentaires_retard_arrivee,object,0.93,302
date,object,0.00,82
retard_moyen_tous_trains_arrivee,float64,0.00,9813
prct_cause_gestion_gare,float64,0.00,1274
prct_cause_materiel_roulant,float64,0.00,1554
prct_cause_gestion_trafic,float64,0.00,1659
prct_cause_infra,float64,0.00,1598
prct_cause_externe,float64,0.00,1726


### 3.4 Schema-as-contract

*TODO (Exercise 3 of Session 6)*: write a `pa.DataFrameSchema` for the columns you keep. Requirements: every kept column with a dtype, `strict=True`, **at least 4 domain checks**, `nullable=True` only where justified in a comment. Validate the raw data with `lazy=True` and keep the failure summary for your report.

Columns to look at first (confirm the exact names in `df_raw.columns`): `date`, `service`, `gare_depart`, `gare_arrivee`, `duree_moyenne`, `nb_train_prevu`, `nb_annulation`, `nb_train_retard_arrivee`, `retard_moyen_arrivee` and the `prct_cause_*` columns.

In [ ]:
# TODO: select the columns you keep, then write the contract
KEEP = list(df_raw.columns)          # TODO: replace by your selection (and log it as a decision)
df_sel = df_raw[KEEP].copy()

schema = None                        # TODO: replace None by pa.DataFrameSchema({...}, strict=True)

if schema is not None:
    try:
        schema.validate(df_sel, lazy=True)
        print("✅ raw data passes the contract")
    except pa.errors.SchemaErrors as err:
        fails = err.failure_cases
        print(f"❌ {len(fails)} violations")
        display(fails.groupby(["column", "check"]).size().rename("n").reset_index())

---
## 4 · Data quality — measure → decide → log

Every decision gets one `log(...)` call. The table at the end of this section goes into your report.

**What to look for in this dataset:**
- Missing months for some routes (COVID 2020, strikes): is that MNAR? Can you compare years fairly?
- Station names that change spelling over time: duplicates in disguise. Build a mapping and log it.
- Do the `prct_cause_*` shares add up to 100%? Check row by row.
- Means of means: `retard_moyen_arrivee` is already an average — weight by number of trains when aggregating.

In [ ]:
decisions = []
def log(column, issue, n_rows, decision, why):
    decisions.append(dict(column=column, issue=issue, n_rows=n_rows, decision=decision, why=why))

df = df_sel.copy()

### 4.1 Duplicates

In [ ]:
n_exact = int(df.astype(str).duplicated().sum())    # astype(str): some columns may hold dicts
print("exact duplicate rows:", n_exact)
# TODO: decide, apply, and log. Then look for near-duplicates (same entity, different id or spelling).

### 4.2 Impossible values

In [ ]:
# TODO: for each numeric column, what is physically or logically impossible? Count, decide, log.
df.describe().T

### 4.3 Missingness — mechanism before method

In [ ]:
miss = df.isna().mean().sort_values(ascending=False)
miss = miss[miss > 0].rename("share_missing").to_frame()
miss["flag"] = np.where(miss.share_missing > MAX_MISSING_SHARE, "⚠️ high", "")
miss

In [ ]:
# TODO: for the 2–3 most important columns with missing values, test whether missingness
# depends on another observed column (MAR) — e.g.
# df.assign(m=df["<col>"].isna()).groupby("<other col>")["m"].mean()
# Then name the mechanism (MCAR / MAR / MNAR), decide, log.

### 4.4 Outliers — triage, don't delete

In [ ]:
# TODO: pick your key numeric column. Use an IQR fence (on a log scale if it is skewed),
# look at the flagged rows, separate errors from real extremes, decide per group, log.

In [ ]:
decision_table = pd.DataFrame(decisions, columns=["column", "issue", "n_rows", "decision", "why"])
decision_table.to_csv(REPORTS / "decision_table.csv", index=False)
decision_table

---
## 5 · Exploration — every figure gets a caption and a *so what*

In [ ]:
def show(fig, title, caption, so_what, name):
    fig.suptitle(title, x=0.01, ha="left", fontsize=12, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    fig.text(0.01, -0.04, f"{caption}\n→ {so_what}", ha="left", va="top", fontsize=9, wrap=True)
    fig.savefig(REPORTS / f"{name}.png", bbox_inches="tight", dpi=120)
    plt.show()

clean = df.copy()      # TODO: this should be your cleaned data after Section 4

### 5.1 Univariate

In [ ]:
# TODO: one figure showing the distribution of your key variable (ECDF before `.describe()`)
# fig, ax = plt.subplots(figsize=(8, 4))
# ...
# show(fig, "<title that states the finding>", "<what the figure shows>", "<what a modeller should do>", "5_1")

### 5.2 Bivariate

In [ ]:
# TODO: one figure showing your key variable against 1–2 explanatory variables
# fig, ax = plt.subplots(figsize=(8, 4))
# ...
# show(fig, "<title that states the finding>", "<what the figure shows>", "<what a modeller should do>", "5_2")

### 5.3 Multivariate

In [ ]:
# TODO: one figure showing the correlation structure, or a breakdown by two categories
# fig, ax = plt.subplots(figsize=(8, 4))
# ...
# show(fig, "<title that states the finding>", "<what the figure shows>", "<what a modeller should do>", "5_3")

---
## 6 · Findings, what cannot be trusted, next steps

*TODO*

**Three findings** — each with the figure it comes from and what it means for modelling.
1.
2.
3.

**What cannot be trusted** — column, subset, and why.
-
-

**Five testable hypotheses** — variable, direction, population, test.
- H1:
- H2:
- H3:
- H4:
- H5:

### Final contract & fingerprint
*TODO*: write `clean_schema` for the data a modeller receives (it may differ from the raw schema). Then **Restart & Run All twice**: the fingerprint must be identical.

In [ ]:
clean_schema = None                  # TODO: pa.DataFrameSchema({...}, strict=True)
if clean_schema is not None:
    clean_schema.validate(clean, lazy=True)
    print("✅ clean data passes the contract")

clean.to_parquet(DATA / "processed" / "clean.parquet", index=False)
fingerprint = hashlib.sha256(
    pd.util.hash_pandas_object(clean.astype(str), index=False).values.tobytes()).hexdigest()[:12]
print("rows:", len(clean), "· fingerprint:", fingerprint)

> **Reproducibility note.** The fingerprint is stable between runs *thanks to the cache*: the API data changes over time, so a run without the cache (or after `CACHE_HOURS`) can give a different fingerprint. That is expected — record the collection date in Section 1 and in the README, and commit the raw Parquet if it is small enough.